In [23]:
import pandas as pd
import numpy as np
import json
import glob
import csv
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel

import spacy
from nltk.corpus import stopwords
import pyLDAvis
import pyLDAvis.gensim
import pyLDAvis.gensim_models

In [24]:
def load_data(file):
    with open (file, "r", encoding="utf-8") as f:
        data = json.load(f)
    return (data)

def write_data(file, data):
    with open (file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

In [25]:
stopwords = stopwords.words("english")
stopwords.append("be")
print(stopwords)

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

In [26]:
# Load the JSON data from the file
with open("kanan_bag_of_words.json", "r") as file:
    data = json.load(file)

# Get the keys of the JSON object
fields = data.keys()

# Print the fields
print(fields)


dict_keys(['BagOfWords'])


In [32]:
data = load_data("kanan_bag_of_words.json")['BagOfWords']
print (data[0][0:90])
# print (data[1][0:90])

good,evening,hello,how,are,you,it's,great,to,be,back,in,india,i've,been,traveling,the,worl


In [33]:
def lemmatization(texts, allowed_postages=["NOUN", "ADJ", "VERB", "ADV"]):
    nlp = nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    texts_out = []
    for text in data:
        doc = nlp(text)
        new_text = []
        for token in doc:
            if token.pos_ in allowed_postages:
                new_text.append(token.lemma_)
        final = " ".join(new_text)
        texts_out.append(final)
    return (texts_out)

lemmatized_texts = lemmatization(data)
print (lemmatized_texts[0][0:150])

good evening great back in be travel world lot year so good back in need small talk single good thing about look don't know in eye just fucking walk p


In [34]:
import nltk
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download()

showing info https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/index.xml


True

In [64]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = stopwords.words('english')
stop_words.extend(['from', 'write', 'good', 'feel', 'make', 'take', 'thing', 'subject', 're', 'edu', 'use', 'be', 'know', 'go', 'think', 'come', 'see', 'guy', 'say', 'even', 'year', 'one', 'would', 'find', 'get'])
def sent_to_words(sentences):
    for sentence in sentences:
        # deacc=True removes punctuations
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) 
             if word not in stop_words] for doc in texts]

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/alisha/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [65]:
def gen_words(data):
    final = []
    for text in data:
        new = gensim.utils.simple_preprocess(text, deacc=True)
        final.append(new)
    return (final)

data_words = gen_words(lemmatized_texts)

print (data_words[0][0:20])
# print (data_words[1][0:20])

['good', 'evening', 'great', 'back', 'in', 'be', 'travel', 'world', 'lot', 'year', 'so', 'good', 'back', 'in', 'need', 'small', 'talk', 'single', 'good', 'thing']


In [66]:
data_words = remove_stopwords(data_words)
print(data_words[:1][0][:30])

['evening', 'great', 'back', 'travel', 'world', 'lot', 'back', 'need', 'small', 'talk', 'single', 'look', 'eye', 'fucking', 'walk', 'past', 'deserve', 'interaction', 'case', 'country', 'coffee', 'national', 'sport', 'whole', 'walk', 'first', 'thank', 'hot', 'outside', 'hot']


In [67]:
#BIGRAMS AND TRIGRAMS
bigram_phrases = gensim.models.Phrases(data_words, min_count=5, threshold=200)
trigram_phrases = gensim.models.Phrases(bigram_phrases[data_words], threshold=100)

bigram = gensim.models.phrases.Phraser(bigram_phrases)
trigram = gensim.models.phrases.Phraser(trigram_phrases)

def make_bigrams(texts):
    return([bigram[doc] for doc in texts])

def make_trigrams(texts):
    return ([trigram[bigram[doc]] for doc in texts])

data_bigrams = make_bigrams(data_words)
data_bigrams_trigrams = make_trigrams(data_bigrams)

print (data_bigrams_trigrams[0])

['evening', 'great', 'back', 'travel', 'world', 'lot', 'back', 'need', 'small', 'talk', 'single', 'look', 'eye', 'fucking', 'walk', 'past', 'deserve', 'interaction', 'case', 'country', 'coffee', 'national', 'sport', 'whole', 'walk', 'first', 'thank', 'hot', 'outside', 'hot', 'begin', 'speak', 'much', 'time', 'population', 'double', 'small', 'talk', 'big', 'talk', 'talk', 'aunty', 'beta', 'yearly', 'salary', 'away', 'long', 'back', 'different', 'airport', 'silent', 'airport', 'announcement', 'airport', 'anymore', 'reason', 'announcement', 'announcement', 'lady', 'gentleman', 'want', 'let', 'happen', 'minute', 'prevent', 'noise', 'pollution', 'snicker', 'big', 'problem', 'running', 'water', 'least', 'die', 'quietly', 'prevent', 'noise', 'pollution', 'announcement', 'people', 'still', 'catch', 'flight', 'send', 'airline', 'personally', 'scream', 'face', 'noise', 'pollution', 'prevent', 'simple', 'announcement', 'goair', 'goa', 'passenger', 'report', 'gate', 'number', 'walk', 'around', 'go

In [68]:
# TF-IDF REMOVAL
from gensim.models import TfidfModel

id2word = corpora.Dictionary(data_bigrams_trigrams)

texts = data_bigrams_trigrams

corpus = [id2word.doc2bow(text) for text in texts]
print (corpus[0][0:20])

tfidf = TfidfModel(corpus, id2word=id2word)

low_value = 0.03
words = []
words_missing_in_tfidf = []
for i in range (0, len(corpus)):
    bow = corpus[i]
    low_value_words = [] # reinitialization to be safe, you can skip this
    tfidf_ids = [id for id, value in bow]
    bow_ids = [id for id, value in bow]
    low_value_words = [id for id, value in tfidf[bow] if value < low_value]
    drops = low_value_words+words_missing_in_tfidf
    for item in drops:
        words.append(id2word[item])
    words_missing_in_tfidf = [id for id in bow_ids if id not in tfidf_ids] # the words with tf-idf score 0 will be missing
    
    new_bow = [b for b in bow if b[0] not in low_value_words and b[0] not in words_missing_in_tfidf]
    corpus[i] = new_bow
    

[(0, 1), (1, 1), (2, 1), (3, 2), (4, 9), (5, 1), (6, 3), (7, 1), (8, 8), (9, 1), (10, 2), (11, 1), (12, 1), (13, 1), (14, 2), (15, 1), (16, 2), (17, 4), (18, 1), (19, 2)]


In [69]:
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus, 
                                           id2word=id2word,
                                           num_topics=2,
                                           random_state=100,
                                           update_every=1,
                                           chunksize=100,
                                           passes=10,
                                           alpha="auto")

In [70]:
lda_model.print_topics()

[(0,
  '0.012*"time" + 0.011*"letter" + 0.009*"goal" + 0.009*"play" + 0.009*"people" + 0.008*"friend" + 0.008*"dog" + 0.007*"band" + 0.007*"stab" + 0.006*"song"'),
 (1,
  '0.001*"letter" + 0.001*"time" + 0.001*"play" + 0.001*"band" + 0.001*"goal" + 0.001*"dog" + 0.001*"friend" + 0.001*"people" + 0.001*"back" + 0.001*"stab"')]

In [73]:
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model, corpus, id2word, mds="mmds", R=10)
vis

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0     -0.011419  0.036048       1        1  99.958909
1      0.011419 -0.036048       2        1   0.041091, topic_info=         Term       Freq      Total Category  logprob  loglift
444    letter  30.000000  30.000000  Default  10.0000  10.0000
842      time  31.000000  31.000000  Default   9.0000   9.0000
601      play  24.000000  24.000000  Default   8.0000   8.0000
66       band  19.000000  19.000000  Default   7.0000   7.0000
213       dog  19.000000  19.000000  Default   6.0000   6.0000
335      goal  24.000000  24.000000  Default   5.0000   5.0000
62       back  16.000000  16.000000  Default   4.0000   4.0000
306    friend  21.000000  21.000000  Default   3.0000   3.0000
584    people  22.000000  22.000000  Default   2.0000   2.0000
763      stab  17.000000  17.000000  Default   1.0000   1.0000
842      time  31.843524  31.844983   Topic1  -4.4196   0.0004
444    letter  30.133947  30.135419   Topic1  -4.4747   0.0004
335      goal  24.202598  24.203975   Topic1  -4.6939   0.0004
601      play  24.192722  24.194126   Topic1  -4.6943   0.0004
584    people  22.513991  22.515325   Topic1  -4.7663   0.0004
306    friend  21.661474  21.662809   Topic1  -4.8049   0.0003
213       dog  19.945665  19.947028   Topic1  -4.8874   0.0003
66       band  19.085182  19.086567   Topic1  -4.9315   0.0003
763      stab  17.405529  17.406847   Topic1  -5.0236   0.0003
752      song  16.562121  16.563415   Topic1  -5.0733   0.0003
614  pregnant   0.001149   1.274534   Topic2  -6.8530   0.7853
516      move   0.001148   1.274617   Topic2  -6.8531   0.7850
858     trick   0.001148   1.274626   Topic2  -6.8532   0.7850
452  likelike   0.001148   1.274680   Topic2  -6.8533   0.7848
547     nurse   0.001148   1.274688   Topic2  -6.8533   0.7848
347      hair   0.001148   1.274729   Topic2  -6.8534   0.7847
930      yell   0.001148   1.274784   Topic2  -6.8535   0.7845
870     uncle   0.001148   1.274785   Topic2  -6.8535   0.7845
764   stammer   0.001148   1.274814   Topic2  -6.8536   0.7844
803   surgery   0.001148   1.274816   Topic2  -6.8536   0.7844
444    letter   0.001472  30.135419   Topic2  -6.6047  -2.1296
842      time   0.001459  31.844983   Topic2  -6.6134  -2.1935
66       band   0.001385  19.086567   Topic2  -6.6656  -1.7337
601      play   0.001404  24.194126   Topic2  -6.6523  -1.9576
213       dog   0.001364  19.947028   Topic2  -6.6812  -1.7935
335      goal   0.001377  24.203975   Topic2  -6.6712  -1.9769
62       back   0.001324  16.552422   Topic2  -6.7111  -1.6368
292     first   0.001302  12.298540   Topic2  -6.7278  -1.3565
306    friend   0.001334  21.662809   Topic2  -6.7029  -1.8977
763      stab   0.001318  17.406847   Topic2  -6.7152  -1.6912
584    people   0.001334  22.515325   Topic2  -6.7031  -1.9365, token_table=      Topic      Freq      Term
term                           
62        1  1.027040      back
66        1  0.995465      band
213       1  1.002656       dog
292       1  0.975726     first
306       1  1.015565    friend
335       1  0.991573      goal
347       1  0.784481      hair
444       1  0.995506    letter
452       1  0.784511  likelike
516       1  0.784549      move
547       1  0.784506     nurse
584       1  1.021526    people
601       1  0.991976      play
614       1  0.784600  pregnant
752       1  1.026358      song
763       1  0.976627      stab
764       1  0.784428   stammer
803       1  0.784427   surgery
842       1  1.004868      time
858       1  0.784544     trick
870       1  0.784446     uncle
930       1  0.784447      yell, R=10, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 2])

In [72]:
import json
import re

# Read the JSON file
with open('kanan_transcripts.json', 'r') as file:
    data = json.load(file)

# Extract the transcripts
transcripts = data['Transcripts']

# Define a regular expression pattern to match sentences containing the word 'friend'
pattern = re.compile(r'\b(?:friend)\b', flags=re.IGNORECASE)

# Find sentences containing the word 'friend' and store them in a list
sentences_with_friend = []
for transcript in transcripts:
    sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', transcript)
    for sentence in sentences:
        if re.search(pattern, sentence):
            sentences_with_friend.append(sentence)

# Print or process the sentences containing the word 'friend'
for sentence in sentences_with_friend:
    print(sentence)


And all the practice letters you write are addressed to a fictional friend who lives in boarding school.
It's always, Write a letter to your friend in boarding school...
Who is that kind of friend...
[superior tone] Dear friend...
Option A--Write a letter to your friend in boarding school wishing them a happy birthday, giving them a present.
to your friend who has just lost both of their parents.
In the letter, I wrote, Dear friend, I am writing this letter to wish you a happy birthday.
I have a friend.
Now, the last person who stabbed Caesar is his best friend Brutus, and Brutus stabs Caesar in the back for a metaphor.
Do you guys have a friend in your life whose life is worse than yours?
Your friend is like, Hi. You're like, Oh, yeah.
If you don't have a friend like that, it's you.
Friend is sitting inside my house already.
I'm a good friend! That's all I'm really saying.
He calls his friend, Hey, man.
And his friend is like, Dude, fuck! What's wrong with you?
Came home with my frien

In [63]:
import json
import re

# Read the JSON file
with open('kanan_transcripts.json', 'r') as file:
    data = json.load(file)

# Extract the transcripts
transcripts = data['Transcripts']

# Define a regular expression pattern to match sentences containing the word 'friend'
pattern = re.compile(r'\b(?:time)\b', flags=re.IGNORECASE)

# Find sentences containing the word 'friend' and store them in a list
sentences_with_friend = []
for transcript in transcripts:
    sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', transcript)
    for sentence in sentences:
        if re.search(pattern, sentence):
            sentences_with_friend.append(sentence)

# Print or process the sentences containing the word 'friend'
for sentence in sentences_with_friend:
    print(sentence)


In this much time, the population of India has doubled.
It's good time pass, no?
That's an Indian concept--time pass.
Time pass is both an activity and a review...
Time pass.
Oh, time pass.
Like the passage of time.
But in India, we do time pass.
Time will pass.
In the West, they don't have time pass.
It's something you do to pass time.
We're like, Well, we read your writing, and it's time pass.
That's because time is relative.
You see, this whole time, I've been lifting with my hands.
We were studying about a great man at the time.
At the time, we were studying William Shakespeare's Julius Caesar, -which, if you-- -[audience cheers] Okay.
[chuckles] If you haven't read, is time pass.
That was the first time that had ever happened.
That guy won't return our money because you pay for Samantha's time.
Every time he did, I'd go, Bah.
Not like I had so much time.
You think I have free time to be entertaining a dog?
First time in a video game you could do anything you felt like.
But it was 

In [75]:
import json
import re

# Read the JSON file
with open('kanan_transcripts.json', 'r') as file:
    data = json.load(file)

# Extract the transcripts
transcripts = data['Transcripts']

# Define a regular expression pattern to match sentences containing the word 'friend'
pattern = re.compile(r'\b(?:stab)\b', flags=re.IGNORECASE)

# Find sentences containing the word 'friend' and store them in a list
sentences_with_friend = []
for transcript in transcripts:
    sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', transcript)
    for sentence in sentences:
        if re.search(pattern, sentence):
            sentences_with_friend.append(sentence)

# Print or process the sentences containing the word 'friend'
for sentence in sentences_with_friend:
    print(sentence)


Seven people stab Julius Caesar 23 times.
